## Step 1: Load Order Intake Data

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [2]:
FILE_PATH = "/Users/apple/AI Matics/Fibro/ROL Project/data/ROL Working.xlsx"

df = pd.read_excel(FILE_PATH, sheet_name="Data")

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")

display(df.head())

Rows    : 4,817
Columns : 17


,OA Date,Item Code,Sum of Sales_Qty,Year,Week No,Week,Week-Year,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,2025-01-02,4960.85.125.250,4,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,4960.85.048.150,6,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-22,NaN,Week 10-2025
2,2025-01-02,4960.85.075.150,2,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-23,NaN,Week 10-2026
3,2025-01-02,4960.85.100.150,1,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-24,NaN,Week 11-2025
4,2025-01-02,4960.85.048.100,2,2025,2025-1,Week 1,Week 1-2025,NaN,NaN,NaN,NaN,18.0,NaN,NaN,2024-25,NaN,Week 11-2026


## Step 2: Data Preparation

In [3]:
df["OA Date"] = pd.to_datetime(df["OA Date"])

df["Year"] = df["OA Date"].dt.isocalendar().year.astype(int)
df["Week"] = df["OA Date"].dt.isocalendar().week.astype(int)

df = df.sort_values("OA Date").reset_index(drop=True)

display(df.head())

,OA Date,Item Code,Sum of Sales_Qty,Year,Week No,Week,Week-Year,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
0,2025-01-02,4960.85.125.250,4,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,4960.85.048.150,6,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-22,NaN,Week 10-2025
2,2025-01-02,4960.85.075.150,2,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-23,NaN,Week 10-2026
3,2025-01-02,4960.85.100.150,1,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024-24,NaN,Week 11-2025
4,2025-01-02,4960.85.048.100,2,2025,2025-1,1,Week 1-2025,NaN,NaN,NaN,NaN,18.0,NaN,NaN,2024-25,NaN,Week 11-2026


## Step 3: Aggregate Weekly Demand

In [4]:
weekly = (
    df.groupby(
        ["Item Code", "Year", "Week"],
        as_index=False
    )["Sum of Sales_Qty"]
    .sum()
)

weekly.rename(
    columns={"Sum of Sales_Qty": "Weekly Demand"},
    inplace=True
)

display(weekly.head())

,Item Code,Year,Week,Weekly Demand
0,4960.85.028.075,2025,6,2
1,4960.85.028.075,2025,23,2
2,4960.85.028.075,2025,29,48
3,4960.85.028.075,2025,31,4
4,4960.85.028.075,2025,32,2


## Step 4: Generate Product Summary

In [5]:
summary = (
    weekly.groupby("Item Code", as_index=False)
    .agg(
        Total_Sales=("Weekly Demand", "sum"),
        Average_Weekly_Demand=("Weekly Demand", "mean"),
        Maximum_Weekly_Demand=("Weekly Demand", "max"),
        Number_of_Weeks=("Weekly Demand", "count")
    )
)

display(summary.head())

,Item Code,Total_Sales,Average_Weekly_Demand,Maximum_Weekly_Demand,Number_of_Weeks
0,4960.85.028.075,231,8.555556,48,27
1,4960.85.028.100,24,3.428571,6,7
2,4960.85.028.150,42,8.400000,23,5
3,4960.85.038.050,2,2.000000,2,1
4,4960.85.038.075,377,10.189189,43,37


## Step 5: Apply Dynamic Mode Logic

In [6]:
mode_qty = (
    weekly.groupby("Item Code")["Weekly Demand"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 1)
    .rename("Mode_Qty")
)
summary = summary.merge(
    mode_qty,
    on="Item Code",
    how="left"
)

display(summary.head())

#save summary to CSV
summary.to_csv("weekly_demand_summary.csv", index=False)


,Item Code,Total_Sales,Average_Weekly_Demand,Maximum_Weekly_Demand,Number_of_Weeks,Mode_Qty
0,4960.85.028.075,231,8.555556,48,27,2
1,4960.85.028.100,24,3.428571,6,7,2
2,4960.85.028.150,42,8.400000,23,5,8
3,4960.85.038.050,2,2.000000,2,1,2
4,4960.85.038.075,377,10.189189,43,37,4


## Step 6: Select Product for ROL Calculation

In [10]:


# item_code = "4960.85.150.150"
# item_code = "4960.85.075.125"
# item_code = "4960.85.058.100"
# item_code = "4960.85.125.125"
# item_code = "4960.85.100.150"
item_code = "4960.85.075.100"

item_summary = summary.loc[
    summary["Item Code"] == item_code
]

display(item_summary)

,Item Code,Total_Sales,Average_Weekly_Demand,Maximum_Weekly_Demand,Number_of_Weeks,Mode_Qty
20,4960.85.075.100,3320,46.111111,248,72,18


## Step 7: Extract Weekly Demand

In [11]:
weekly_item = (
    weekly.loc[
        weekly["Item Code"] == item_code,
        ["Year", "Week", "Weekly Demand"]
    ]
    .copy()
)

display(weekly_item)

,Year,Week,Weekly Demand
575,2025,2,54
576,2025,3,48
577,2025,4,28
578,2025,5,4
579,2025,6,69
...,...,...,...
642,2026,18,71
643,2026,19,25
644,2026,20,24
645,2026,21,46


In [12]:
bin_size = int(item_summary["Mode_Qty"].iloc[0])

if bin_size <= 0:
    bin_size = 1

print(f"Bin Size : {bin_size}")

Bin Size : 18


## Step 9: Create Demand Intervals

In [13]:
max_demand = weekly_item["Weekly Demand"].max()

bins = [(0, 0)]

start = 1

while start <= max_demand:
    end = start + bin_size - 1
    bins.append((start, end))
    start += bin_size

bins

[(0, 0),
 (1, 18),
 (19, 36),
 (37, 54),
 (55, 72),
 (73, 90),
 (91, 108),
 (109, 126),
 (127, 144),
 (145, 162),
 (163, 180),
 (181, 198),
 (199, 216),
 (217, 234),
 (235, 252)]

## Step 10: Generate Frequency Distribution

In [14]:
frequency = []

for lower, upper in bins:

    if lower == 0:
        count = 0
    else:
        count = (
            weekly_item["Weekly Demand"]
            .between(lower, upper)
            .sum()
        )

    frequency.append(
        {
            "Lower": lower,
            "Upper": upper,
            "Frequency": count
        }
    )

frequency_df = pd.DataFrame(frequency)

display(frequency_df)

,Lower,Upper,Frequency
0,0,0,0
1,1,18,14
2,19,36,20
3,37,54,19
4,55,72,8
5,73,90,5
6,91,108,2
7,109,126,2
8,127,144,0
9,145,162,1


## Step 11: Include Zero-Demand Weeks

In [15]:
# TOTAL_WEEKS = 74

TOTAL_WEEKS = (
    df["Year"].astype(str) + "-" + df["Week"].astype(str)
).nunique()

print(f"Total Weeks : {TOTAL_WEEKS}")
non_zero_frequency = frequency_df.loc[1:, "Frequency"].sum()

frequency_df.loc[0, "Frequency"] = (
    TOTAL_WEEKS - non_zero_frequency
)

display(frequency_df)

Total Weeks : 74


,Lower,Upper,Frequency
0,0,0,2
1,1,18,14
2,19,36,20
3,37,54,19
4,55,72,8
5,73,90,5
6,91,108,2
7,109,126,2
8,127,144,0
9,145,162,1


## Step 12: Calculate Probability Distribution

In [16]:
frequency_df["Contribution"] = (
    frequency_df["Frequency"] / TOTAL_WEEKS
)

frequency_df["Cum Probability"] = (
    frequency_df["Contribution"].cumsum()
)

frequency_df["Mid Point"] = (
    frequency_df["Lower"] +
    frequency_df["Upper"]
) / 2

frequency_df["Weighted Sum"] = (
    frequency_df["Mid Point"] *
    frequency_df["Contribution"]
)

display(frequency_df)

,Lower,Upper,Frequency,Contribution,Cum Probability,Mid Point,Weighted Sum
0,0,0,2,0.027027,0.027027,0.0,0.000000
1,1,18,14,0.189189,0.216216,9.5,1.797297
2,19,36,20,0.270270,0.486486,27.5,7.432432
3,37,54,19,0.256757,0.743243,45.5,11.682432
4,55,72,8,0.108108,0.851351,63.5,6.864865
5,73,90,5,0.067568,0.918919,81.5,5.506757
6,91,108,2,0.027027,0.945946,99.5,2.689189
7,109,126,2,0.027027,0.972973,117.5,3.175676
8,127,144,0,0.000000,0.972973,135.5,0.000000
9,145,162,1,0.013514,0.986486,153.5,2.074324


## Step 13: Calculate Weekly Demand Statistics

In [17]:
average_weekly_demand = frequency_df["Weighted Sum"].sum()

SERVICE_LEVEL = 0.85

below = frequency_df[
    frequency_df["Cum Probability"] < SERVICE_LEVEL
].iloc[-1]

above = frequency_df[
    frequency_df["Cum Probability"] >= SERVICE_LEVEL
].iloc[0]

fraction = (
    (SERVICE_LEVEL - below["Cum Probability"]) /
    (above["Cum Probability"] - below["Cum Probability"])
)
print(f"Below : {below}")
print(f"Above : {above}")
print(f"Fraction : {fraction:.4f}")
d_max_week = (
    below["Upper"] +
    fraction * (above["Upper"] - below["Upper"])
)

# Round to the nearest whole number
d_max_week = round(d_max_week)
average_weekly_demand = round(average_weekly_demand)

print(f"Average Weekly Demand : {average_weekly_demand:.2f}")
print(f"Dmax / Week : {d_max_week}")


Below : Lower              37.000000
Upper              54.000000
Frequency          19.000000
Contribution        0.256757
Cum Probability     0.743243
Mid Point          45.500000
Weighted Sum       11.682432
Name: 3, dtype: float64
Above : Lower              55.000000
Upper              72.000000
Frequency           8.000000
Contribution        0.108108
Cum Probability     0.851351
Mid Point          63.500000
Weighted Sum        6.864865
Name: 4, dtype: float64
Fraction : 0.9875
Average Weekly Demand : 45.00
Dmax / Week : 72


In [18]:
# #without interpolation

# average_weekly_demand = frequency_df["Weighted Sum"].sum()

# SERVICE_LEVEL = 0.85

# closest_row = frequency_df.iloc[
#     (frequency_df["Cum Probability"] - SERVICE_LEVEL).abs().argmin()
# ]

# d_max_week = round(closest_row["Upper"])
# average_weekly_demand = round(average_weekly_demand)

# print(f"Average Weekly Demand : {average_weekly_demand}")
# print(f"Dmax / Week : {d_max_week}")
# print(f"Selected Cum Probability : {closest_row['Cum Probability']:.4f}")

## Step 14: Convert Weekly Demand to Monthly Demand

In [19]:
average_monthly_demand = average_weekly_demand * 4

d_max_month = d_max_week * 4

print(f"Average Monthly Demand : {average_monthly_demand:.2f}")
print(f"Dmax / Month : {d_max_month}")

Average Monthly Demand : 180.00
Dmax / Month : 288


## Step 15: Calculate Safety Stock

In [20]:
LEAD_TIME = 4

safety_stock = (
    d_max_week -
    average_weekly_demand
) * LEAD_TIME

print(f"Safety Stock : {safety_stock:.2f}")

Safety Stock : 108.00


## Step 16: Calculate Reorder Level

In [21]:
rol = (
    average_monthly_demand + safety_stock)

print(f"Reorder Level : {rol:.2f}")

Reorder Level : 288.00


## Final Result

In [22]:
result = pd.DataFrame(
    {
        "Metric": [
            "Item Code",
            "Average Weekly Demand",
            "Average Monthly Demand",
            "Maximum Weekly Demand (Dmax)",
            "Maximum Monthly Demand",
            "Safety Stock",
            "Reorder Level"
        ],
        "Value": [
            item_code,
            round(average_weekly_demand, 2),
            round(average_monthly_demand, 2),
            d_max_week,
            d_max_month,
            round(safety_stock, 2),
            round(rol, 2)
        ]
    }
)

display(result)

,Metric,Value
0,Item Code,4960.85.075.100
1,Average Weekly Demand,45
2,Average Monthly Demand,180
3,Maximum Weekly Demand (Dmax),72
4,Maximum Monthly Demand,288
5,Safety Stock,108
6,Reorder Level,288
